Setup

In [2]:
import os, sys, json
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
sys.path.insert(0, os.path.abspath("../src"))   # so we can import nlp_utils

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from nlp_utils import ctfidf_keywords, split_sentences, share_by_year, growth

df = pd.read_pickle("../artifacts/reviews_topics.pkl")
emb = np.load("../artifacts/embeddings.npy")
model = SentenceTransformer("all-MiniLM-L6-v2")
print(df.shape, emb.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(50001, 16) (50001, 384)


most negative topic

In [3]:
topic_neg = (df.groupby("topic_label")
               .agg(n=("Score", "size"), pct_negative=("Score", lambda s: (s <= 2).mean()))
               .sort_values("pct_negative", ascending=False).round(3))
topic_neg

,n,pct_negative
topic_label,,
Packaging & delivery,2593,0.426
Price & value vs. stores,1613,0.252
Baby formula & milk,1993,0.214
"Drinks, juices & sweeteners",3715,0.172
Gifts & orders,1988,0.152
Chocolate & cocoa,1729,0.147
General taste & quality,3928,0.145
Dog food,1191,0.141
Coffee & K-cups,4834,0.139


Complaint themes, from clustering only the 1–2★ reviews

In [4]:
neg_mask = (df.Score <= 2).values
neg = df[neg_mask].reset_index(drop=True)
neg_emb = emb[neg_mask]
print(f"{len(neg):,} complaint reviews ({neg_mask.mean():.1%} of sample)")

KC = 12
kmc = KMeans(n_clusters=KC, n_init=5, random_state=42).fit(neg_emb)
neg["complaint_id"] = kmc.labels_
c_cent = kmc.cluster_centers_ / np.linalg.norm(kmc.cluster_centers_, axis=1, keepdims=True)

c_keywords = ctfidf_keywords(neg.text_for_embedding.values, neg.complaint_id.values)
for c in range(KC):
    n = (neg.complaint_id == c).sum()
    print(f"{c:2d} | {n:5d} | {', '.join(c_keywords[c][:8])}")

7,247 complaint reviews (14.5% of sample)
 0 |   689 | chocolate, cookies, bars, candy, cookie, sugar, bar, sweet
 1 |   490 | hair, money, work, did, bottle, plant, does, disappointed
 2 |   714 | box, order, ordered, received, cans, item, shipping, bag
 3 |   281 | food, cat, cats, dog, eat, chicken, dogs, ingredients
 4 |   640 | sugar, ingredients, free, organic, oil, food, natural, gluten
 5 |   399 | tea, green, teas, box, drink, bags, water, bag
 6 |   427 | price, box, pack, store, shipping, oz, cost, expensive
 7 |   807 | stale, eat, bag, chips, bad, hard, box, nuts
 8 |   796 | sauce, soup, cheese, noodles, food, chicken, pasta, salt
 9 |   729 | coffee, cup, cups, roast, beans, keurig, pods, bitter
10 |   351 | dog, dogs, treats, toy, china, treat, food, chicken
11 |   924 | water, drink, sugar, bad, sweet, juice, did, tasted


Example complaints per theme

In [5]:
def complaint_examples(c, n=3):
    idx = np.where(neg.complaint_id == c)[0]
    best = idx[np.argsort(-(neg_emb[idx] @ c_cent[c]))[:n]]
    for _, r in neg.iloc[best].iterrows():
        print(f"  {r.Score}★ {r.Summary} — {r.Text[:180]}")

for c in range(KC):
    print(f"\n=== Theme {c}: {', '.join(c_keywords[c][:5])}")
    complaint_examples(c)


=== Theme 0: chocolate, cookies, bars, candy, cookie
  2★ Recieved this item damaged... — It's hard for me to decide how to rate this product. Since I have had this flavor before, I know that they are tasty and my 10 yr old loves them. However, the chocolate bars were d
  1★ Just plain BAD — I hate giving one star reviews. In my quest for edible, sugar-free chocolate, I'm trying all I can find. I get really excited when one is actually *tasty*! This one, made with stev
  1★ Disappointing — I was very disappointed in the taste of these cookies. Very bland, not sweet enough, disappointing texture and not enough chocolate chips. I ended up throwing them in the trash. Wh

=== Theme 1: hair, money, work, did, bottle
  1★ Not worth money — Purchased this product and was not happy with it at all. It did not even taste or smell like lemon. I was really disappointed.
  1★ DON'T BUY, SMELLS TERRIBLE — DON'T BUY, IT SMELLS TERRIBLE, NOT EDIBLE. I'M SO REGRET. I THREW IT INTO TRASHCAN. THIS SHOUL

Label the complaint themes

In [6]:
c_labels = {
    0:  "Sweets disappointments (chocolate, cookies, candy)",
    1:  "Didn't work / waste of money (non-food)",
    2:  "Order problems: wrong, missing or damaged items",
    3:  "Pet food quality & ingredients",
    4:  "Ingredient & health-claim concerns",
    5:  "Tea disappointments",
    6:  "Overpriced vs. store / size",
    7:  "Stale or hard snacks (chips, nuts)",
    8:  "Savory meals: sauces, soups, pasta",
    9:  "Coffee & K-cup problems (bitter, pods)",
    10: "Dog treats: China sourcing & safety",
    11: "Drinks: bad taste or too sweet",
}
neg["complaint_label"] = neg.complaint_id.map(c_labels)

Complaint drivers, using zero-shot classification at sentence level

In [7]:
ASPECTS = {
    "Taste & flavor": ["it tastes terrible", "the flavor is bland and artificial", "bitter aftertaste",
                       "too sweet", "the flavor was disappointing"],
    "Freshness & expiry": ["the product was stale", "it arrived expired", "past the expiration date",
                           "tasted old and rancid", "it was moldy"],
    "Packaging & shipping damage": ["the box arrived crushed", "the package was damaged in shipping",
                                    "the bag was torn open and leaking", "cans arrived dented",
                                    "it melted during shipping"],
    "Price & value": ["way too expensive", "overpriced for what you get",
                      "much cheaper at the grocery store", "not worth the money"],
    "Quantity & size": ["the package is much smaller than expected", "very few pieces in the bag",
                        "less product than advertised"],
    "Ingredients & health": ["contains too much sugar", "full of artificial ingredients",
                             "high fructose corn syrup", "the ingredients are unhealthy"],
    "Made in China / safety": ["made in china", "ingredients sourced from china",
                               "I worry about safety", "contaminated product"],
    "Got sick / allergic reaction": ["it made me sick", "I got a stomach ache", "gave me diarrhea",
                                     "I had an allergic reaction"],
    "Pet got sick / refused": ["my dog got sick after eating it", "my cat vomited",
                               "the dog would not eat it", "my cat refuses to touch it"],
    "Recipe / formula changed": ["they changed the recipe", "the new formula is worse",
                                 "it does not taste like it used to"],
    "Not as described": ["not what was pictured", "the description is misleading",
                         "received a different product", "wrong flavor was sent"],
    "Seller & delivery service": ["shipping took forever", "the seller never responded",
                                  "customer service was unhelpful", "order never arrived"],
    "Texture & consistency": ["the texture is gritty", "too hard to chew", "soggy and mushy",
                              "chalky texture"],
    "Coffee pods / brewing": ["the k-cups leaked in my keurig", "the pods clog the machine",
                              "coffee is weak and watery"],
}

names = list(ASPECTS)
a_cent = np.vstack([model.encode(ASPECTS[n], normalize_embeddings=True).mean(0) for n in names])
a_cent /= np.linalg.norm(a_cent, axis=1, keepdims=True)

# split complaints into sentences (~5 min to embed on CPU)
sent_rows = [(i, s) for i, t in enumerate(neg.Text) for s in split_sentences(t)]
print(f"{len(sent_rows):,} sentences")
sent_emb = model.encode([s for _, s in sent_rows], batch_size=128,
                        show_progress_bar=True, normalize_embeddings=True)
np.save("../artifacts/neg_sentence_emb.npy", sent_emb)

38,144 sentences


Batches:   0%|          | 0/298 [00:00<?, ?it/s]

Tagging the sentences and rolling the tags up to reviews

In [8]:
THRESHOLD = 0.45   # raise for stricter tagging, lower for more coverage
sims = sent_emb @ a_cent.T
best = sims.argmax(1)
best_sim = sims.max(1)

review_aspects = [set() for _ in range(len(neg))]
evidence = {n: [] for n in names}
for (ri, s), b, sv in zip(sent_rows, best, best_sim):
    if sv >= THRESHOLD:
        review_aspects[ri].add(names[b])
        evidence[names[b]].append((sv, s, ri))

neg["aspects"] = [sorted(a) for a in review_aspects]
print(f"Complaints with ≥1 driver: {(neg.aspects.str.len() > 0).mean():.1%}")

drivers = pd.DataFrame({
    "driver": names,
    "n_reviews": [neg.aspects.map(lambda a, n=n: n in a).sum() for n in names],
})
drivers["pct_of_complaints"] = (drivers.n_reviews / len(neg)).round(3)
drivers.sort_values("n_reviews", ascending=False)

Complaints with ≥1 driver: 77.1%


,driver,n_reviews,pct_of_complaints
0,Taste & flavor,3013,0.416
5,Ingredients & health,1004,0.139
3,Price & value,738,0.102
13,Coffee pods / brewing,597,0.082
11,Seller & delivery service,520,0.072
2,Packaging & shipping damage,490,0.068
8,Pet got sick / refused,430,0.059
1,Freshness & expiry,410,0.057
9,Recipe / formula changed,325,0.045
12,Texture & consistency,245,0.034


Checking the evidence

In [9]:
for n in drivers.sort_values("n_reviews", ascending=False).driver[:6]:
    print(f"\n=== {n}")
    for sv, s, ri in sorted(evidence[n], reverse=True)[:3]:
        print(f"  ({sv:.2f}) {s[:150]}")


=== Taste & flavor
  (0.80) Horrible flavor, horrible taste.
  (0.80) The flavor is bland.
  (0.79) It tasted so bland.

=== Ingredients & health
  (0.76) high fructose corn syrup and sugar.
  (0.76) Has high fructose corn syrup aka sugar and quite a bit.
  (0.74) but the ingredients list 'high fructose corn syrup'.

=== Price & value
  (0.76) PRICE TOO EXPENSIVE.!
  (0.73) This item is way over priced.
  (0.73) this is way too expensive.

=== Coffee pods / brewing
  (0.75) Over half of the k cups have exploded in our Keurig.
  (0.74) I have Coffee People K Cups and they do not do this.
  (0.74) I must've gotten an older batch of the k-cups because they apparently explode in the keurig machine if the top of the k-cup is flat instead of bubbled

=== Seller & delivery service
  (0.69) It took them eight days to respond to my order and ship it, so that is a little slower than some merchants I've done business with through Amazon.
  (0.68) Took 3 days to ship.
  (0.67) Contacted Amazon an

In [15]:
ct = (neg[["complaint_label", "aspects"]]
      .explode("aspects")
      .dropna()
      .reset_index(drop=True)          # ← new: make the index unique again
      .pipe(lambda d: pd.crosstab(d.complaint_label, d.aspects, normalize="index"))
      .round(2))
ct.style.background_gradient(cmap="Reds", axis=1).format("{:.0%}")

aspects,Coffee pods / brewing,Freshness & expiry,Got sick / allergic reaction,Ingredients & health,Made in China / safety,Not as described,Packaging & shipping damage,Pet got sick / refused,Price & value,Quantity & size,Recipe / formula changed,Seller & delivery service,Taste & flavor,Texture & consistency
complaint_label,,,,,,,,,,,,,,
"Coffee & K-cup problems (bitter, pods)",46%,2%,1%,3%,0%,1%,2%,0%,6%,1%,2%,4%,33%,0%
Didn't work / waste of money (non-food),1%,9%,3%,4%,1%,3%,10%,5%,26%,2%,2%,21%,11%,1%
Dog treats: China sourcing & safety,0%,2%,1%,7%,12%,1%,4%,46%,10%,2%,2%,5%,6%,2%
Drinks: bad taste or too sweet,2%,2%,4%,16%,0%,0%,1%,0%,5%,1%,5%,2%,61%,1%
Ingredient & health-claim concerns,0%,2%,5%,46%,5%,2%,2%,1%,6%,1%,5%,3%,22%,1%
"Order problems: wrong, missing or damaged items",2%,8%,1%,3%,1%,4%,31%,1%,8%,7%,1%,22%,11%,1%
Overpriced vs. store / size,3%,1%,0%,5%,0%,2%,3%,0%,59%,7%,1%,7%,11%,0%
Pet food quality & ingredients,0%,2%,2%,14%,0%,0%,4%,54%,6%,1%,5%,3%,8%,2%
"Savory meals: sauces, soups, pasta",0%,4%,3%,11%,1%,0%,2%,2%,5%,1%,8%,3%,54%,5%


Emerging patterns, meaning which complaints are growing

In [16]:
# Complaint themes: share of all complaints per year
theme_share = share_by_year(neg, "complaint_label")
theme_growth = growth(theme_share).sort_values(ascending=False)

# Drivers: share of complaints mentioning each driver per year
expl = neg[["year", "aspects"]].explode("aspects").dropna()
n_per_year = neg.groupby("year").size()
driver_share = (expl.groupby(["aspects", "year"]).size().unstack(fill_value=0)
                .div(n_per_year, axis=1).reindex(columns=range(2006, 2013), fill_value=0))
driver_growth = growth(driver_share).sort_values(ascending=False)

print("Fastest-growing complaint drivers (2011-12 vs 2008-10):")
print(driver_growth.head(6))
print("\nFastest-growing complaint themes:")
print(theme_growth.head(6))

Fastest-growing complaint drivers (2011-12 vs 2008-10):
aspects
Not as described                2.155
Quantity & size                 0.832
Coffee pods / brewing           0.379
Texture & consistency           0.321
Pet got sick / refused          0.239
Got sick / allergic reaction    0.163
dtype: float64

Fastest-growing complaint themes:
complaint_label
Coffee & K-cup problems (bitter, pods)             0.375
Dog treats: China sourcing & safety                0.267
Pet food quality & ingredients                     0.237
Overpriced vs. store / size                        0.163
Order problems: wrong, missing or damaged items    0.063
Ingredient & health-claim concerns                 0.003
dtype: float64


Robustness check

In [22]:
from math import sqrt
from scipy.stats import norm

base_years, recent_years = [2008, 2009, 2010], [2011, 2012]
nb = neg.year.isin(base_years).sum()        # complaints in baseline window
nr = neg.year.isin(recent_years).sum()      # complaints in recent window

rows = []
for d in names:
    has = neg.aspects.map(lambda a, d=d: d in a)
    xb = (has & neg.year.isin(base_years)).sum()
    xr = (has & neg.year.isin(recent_years)).sum()
    pb, pr = xb / nb, xr / nr
    p = (xb + xr) / (nb + nr)
    se = sqrt(p * (1 - p) * (1 / nb + 1 / nr)) or 1e-9
    z = (pr - pb) / se
    rows.append({"driver": d, "n_base": xb, "n_recent": xr,
                 "share_base": round(pb, 3), "share_recent": round(pr, 3),
                 "growth": round((pr - pb) / max(pb, 1e-9), 2),
                 "p_value": round(2 * (1 - norm.cdf(abs(z))), 4)})

sig = pd.DataFrame(rows).sort_values("growth", ascending=False)
sig["significant"] = (sig.p_value < 0.05) & (sig.n_base + sig.n_recent >= 50)
sig
sig.to_csv("../artifacts/driver_significance.csv", index=False)

Chart the top growing drivers

In [23]:
import plotly.express as px
top = driver_growth.head(5).index
plot_df = driver_share.loc[top, 2007:].T.reset_index().melt(id_vars="year", var_name="driver", value_name="share")
px.line(plot_df, x="year", y="share", color="driver", markers=True,
        title="Growing complaint drivers (share of 1–2★ reviews)").update_layout(yaxis_tickformat=".0%").show()

Saving the results

In [24]:
df["complaint_label"] = ""
df.loc[neg_mask, "complaint_label"] = neg.complaint_label.values
df["aspects"] = [[] for _ in range(len(df))]
df.loc[neg_mask, "aspects"] = pd.Series(neg.aspects.values, index=df.index[neg_mask])
df.to_pickle("../artifacts/reviews_enriched.pkl")

out = {
    "complaint_themes": [{"id": c, "label": c_labels[c], "keywords": c_keywords[c],
                          "n": int((neg.complaint_id == c).sum()),
                          "growth": float(theme_growth.get(c_labels[c], 0))} for c in range(KC)],
    "drivers": [{"driver": r.driver, "n_reviews": int(r.n_reviews),
                 "pct_of_complaints": float(r.pct_of_complaints),
                 "growth": float(driver_growth.get(r.driver, 0)),
                 "trend": {int(y): float(v) for y, v in driver_share.loc[r.driver].items()}
                          if r.driver in driver_share.index else {},
                 "quotes": [s for _, s, _ in sorted(evidence[r.driver], reverse=True)[:3]]}
                for r in drivers.itertuples()],
}
with open("../artifacts/complaints.json", "w") as f:
    json.dump(out, f, indent=2)
print("Saved reviews_enriched.pkl and complaints.json")

Saved reviews_enriched.pkl and complaints.json
